# Oráculo: corrección de atenuación Z-PHI

Oráculo de `docs/algorithms/atenuacion-zphi.md`, independiente de cualquier
código Rust (`roadmap.md` §"Método de estudio"): la fórmula de Testud et al.
(2000) se implementa aquí primero. **Verificada 2026-09-18 directamente
contra el paper original** (PDF aportado por el usuario) — coincide con sus
ecuaciones (19), (20), (23) y (24); la corrección de reflectividad es
equivalente por derivación directa a su ecuación (25) (ver
`docs/algorithms/atenuacion-zphi.md` §"Cómo funciona" para el desarrollo
completo). Antes de esa verificación sólo se había contrastado contra la
implementación de Py-ART (`pyart.correct.calculate_attenuation_zphi`,
consultada por búsqueda web); esa comparación se mantiene como contraste
secundario de nombres/rol de coeficientes, no como la verificación
principal.

Cuatro pruebas: (1) la identidad de autoconsistencia del método, algebraica y
verificable sin simulación; (2) recuperación de la Z verdadera en el caso de
modelo acoplado, con medidas de potencia y de ΦDP simuladas de verdad, no el
perfil analítico directo; (3) robustez numérica ante ΔΦDP degenerado
(negativo, cero, extremo); (4) sensibilidad — informativa, no pass/fail — a
un exponente `β` mal asumido, que documenta un hallazgo real: el error NO
siempre se degrada con gracia.

In [ ]:
import numpy as np

rng = np.random.default_rng(20260902)
np.set_printoptions(precision=4, suppress=True)

## Helpers de simulación

Idénticos a los que ya usan `tools/oracles/ruido_y_umbrales.ipynb`
(`generate_cell`, `hildebrand_sekhon`, `noise_floor_estimate`) y
`tools/oracles/kdp_estimacion.ipynb` (`generate_dual_pol_cell`,
`measure_phidp`) — no reinventados aquí, sólo reunidos porque esta página
necesita las dos cadenas: potencia (para CZ) y fase (para ΔΦDP).

In [ ]:
def gaussian_doppler_spectrum(power_s, mean_v, sigma_v, wavelength_m, prt_s, m, wrap=3):
    v_a = wavelength_m / (4.0 * prt_s)
    f_k = np.fft.fftfreq(m, d=prt_s)
    v_k = f_k * wavelength_m / 2.0
    n = np.arange(-wrap, wrap + 1)
    dv = v_k[:, None] - mean_v - n[None, :] * 2.0 * v_a
    acc = np.exp(-0.5 * (dv / sigma_v) ** 2).sum(axis=1)
    return power_s * acc / acc.sum()


def complex_gaussian(rng, variance, size=None):
    sigma = np.sqrt(variance / 2.0)
    return rng.normal(0.0, sigma, size=size) + 1j * rng.normal(0.0, sigma, size=size)


def generate_cell(power_s, mean_v, sigma_v, wavelength_m, prt_s, m, noise_floor, rng, wrap=3):
    spec = gaussian_doppler_spectrum(power_s, mean_v, sigma_v, wavelength_m, prt_s, m, wrap)
    w = complex_gaussian(rng, 1.0, size=m)
    y = np.fft.ifft(w * np.sqrt(spec)) * m
    if noise_floor > 0.0:
        y = y + complex_gaussian(rng, noise_floor, size=m)
    return y


def periodogram(y):
    m = len(y)
    yf = np.fft.fft(y)
    return (np.abs(yf) ** 2) / (m ** 2)


def hildebrand_sekhon(p, navg=1):
    p_sorted = np.sort(p)
    n = len(p_sorted)
    rtest = 1.0 + 1.0 / navg
    nnoise = n
    sum1 = 0.0
    sum2 = 0.0
    for i in range(n):
        pwr = p_sorted[i]
        npts = i + 1
        sum1 += pwr
        sum2 += pwr * pwr
        if npts * sum2 < sum1 * sum1 * rtest:
            nnoise = npts
        else:
            sum1 -= pwr
            sum2 -= pwr * pwr
            break
    return sum1 / nnoise


def noise_floor_estimate(y):
    m = len(y)
    return hildebrand_sekhon(periodogram(y)) * m


def total_power(y):
    return np.mean(np.abs(y) ** 2)


def subtract_noise(r0, n):
    s = r0 - n
    return s if s > 0.0 else None


def generate_dual_pol_cell(power_h, mean_v, sigma_v, wavelength_m, prt_s, m, zdr_db, rho_hv, phidp_deg,
                            noise_floor, rng, wrap=3):
    power_v = power_h / 10.0 ** (zdr_db / 10.0)
    spec_h = gaussian_doppler_spectrum(power_h, mean_v, sigma_v, wavelength_m, prt_s, m, wrap)
    spec_v = gaussian_doppler_spectrum(power_v, mean_v, sigma_v, wavelength_m, prt_s, m, wrap)
    rho = rho_hv * np.exp(1j * np.radians(phidp_deg))
    l21, l22 = np.conj(rho), np.sqrt(1.0 - rho_hv ** 2)
    w1 = complex_gaussian(rng, 1.0, size=m)
    w2 = complex_gaussian(rng, 1.0, size=m)
    wh, wv = w1, l21 * w1 + l22 * w2
    yh = np.fft.ifft(wh * np.sqrt(spec_h)) * m
    yv = np.fft.ifft(wv * np.sqrt(spec_v)) * m
    if noise_floor > 0.0:
        yh = yh + complex_gaussian(rng, noise_floor, size=m)
        yv = yv + complex_gaussian(rng, noise_floor, size=m)
    return yh, yv


def measure_phidp(power_h, mean_v, sigma_v, wavelength_m, prt_s, m, rho_hv, phidp_true_deg, noise_floor, rng):
    yh, yv = generate_dual_pol_cell(power_h, mean_v, sigma_v, wavelength_m, prt_s, m, 1.0,
                                     rho_hv, phidp_true_deg, noise_floor, rng)
    rhv = np.mean(yh * np.conj(yv))
    return np.degrees(np.angle(rhv))


def power_to_dbz(s_linear, range_km, radar_constant_db):
    return 10.0 * np.log10(s_linear) + 20.0 * np.log10(range_km) + radar_constant_db


def dbz_to_power(dbz, range_km, radar_constant_db):
    return 10.0 ** ((dbz - radar_constant_db - 20.0 * np.log10(range_km)) / 10.0)


checks = []

## La fórmula de Testud et al. (2000)

`zphi_specific_attenuation` y `zphi_correct_dbz` son la implementación de
referencia — el mismo código, mismos nombres de variable, que
`crates/attenuation/src/lib.rs` reimplementa en Rust. Ver "Cómo funciona" en
la página del algoritmo para el desarrollo completo; aquí sólo el código.

In [ ]:
def zphi_specific_attenuation(z_dbz, dr_km, beta, a_coef, delta_phidp_deg):
    """Perfil de atenuación específica A(r) [dB/km] -- ver
    docs/algorithms/atenuacion-zphi.md §'Cómo funciona'."""
    z_dbz = np.asarray(z_dbz, dtype=float)
    z_beta = (10.0 ** (z_dbz / 10.0)) ** beta
    n = len(z_beta)
    prefix = np.zeros(n)
    for i in range(1, n):
        prefix[i] = prefix[i - 1] + 0.46 * beta * 0.5 * (z_beta[i - 1] + z_beta[i]) * dr_km
    i_total = prefix[-1]
    assert i_total > 0.0, "el tramo no tiene señal detectable"
    delta_phidp_deg = max(delta_phidp_deg, 0.0)  # censura, no corrige -- ver la página
    c = 10.0 ** (0.1 * beta * a_coef * delta_phidp_deg) - 1.0
    tail = i_total - prefix
    denom = i_total + c * tail
    return z_beta * c / denom


def zphi_correct_dbz(z_dbz, dr_km, beta, a_coef, delta_phidp_deg):
    """Reflectividad corregida [dBZ]: z_dbz + camino bidireccional acumulado de A(r)."""
    a = zphi_specific_attenuation(z_dbz, dr_km, beta, a_coef, delta_phidp_deg)
    n = len(a)
    corr = np.zeros(n)
    for i in range(1, n):
        corr[i] = corr[i - 1] + 0.5 * (a[i - 1] + a[i]) * dr_km * 2.0
    return np.asarray(z_dbz, dtype=float) + corr


BETA = 0.64884  # Gu et al. (2011) vía Py-ART, común a las tres bandas
A_COEF_C_BAND = 0.08  # dB/grado, banda C -- ver la tabla por banda en la página

## Prueba 1 — autoconsistencia

`2·∫A(r)dr` sobre el tramo completo debe coincidir con `a_coef·ΔΦDP`
exactamente (salvo el error de discretización del trapecio), para
CUALQUIER forma del perfil de Z — es una identidad algebraica de la fórmula,
no un resultado de sesgo bajo bajo un modelo particular. Perfil de prueba:
una campana de Z arbitraria, sin relación con ninguna atenuación real.

In [ ]:
DR_KM = 0.150
N_GATES = 200
R_KM = np.arange(N_GATES) * DR_KM


def bell_profile_dbz(peak_dbz=45.0, center_km=15.0, width_km=5.0, floor_dbz=20.0):
    return floor_dbz + peak_dbz * np.exp(-0.5 * ((R_KM - center_km) / width_km) ** 2)


z_dbz_test = bell_profile_dbz()
consistency_ok = True
print(f"{'delta_phidp':>12} {'PIA_est':>10} {'esperado':>10}")
for delta_phidp in [0.5, 5.0, 20.0, 80.0]:
    a = zphi_specific_attenuation(z_dbz_test, DR_KM, BETA, A_COEF_C_BAND, delta_phidp)
    two_way_pia = 2.0 * np.trapezoid(a, dx=DR_KM)
    expected = A_COEF_C_BAND * delta_phidp
    tol = max(0.002 * abs(expected), 1e-3)
    ok = abs(two_way_pia - expected) < tol
    print(f"{delta_phidp:12.1f} {two_way_pia:10.4f} {expected:10.4f}")
    consistency_ok = consistency_ok and ok

checks.append(("Autoconsistencia: 2*integral(A) == a_coef*delta_phidp, cualquier perfil de Z", consistency_ok))

## Prueba 2 — caso de modelo acoplado, con medidas simuladas

La atenuación "verdadera" se genera con la MISMA relación `A = alpha_zA·Z^β`
que asume el método (sólo `β`, nunca `alpha_zA` — ese es justo el punto),
sobre un perfil de Z realista (campana, pico 50 dBZ). El ΔΦDP verdadero es
el que esa atenuación implica vía `a_coef` (`KDP_true = A_true / a_coef`,
`ΦDP_true = 2·∫KDP_true`). Ni la potencia ni la fase se miden "a mano": la
potencia sale de ráfagas simuladas + resta de ruido HS74 + ecuación del radar
(promediada sobre `N_TRIALS_POWER` ráfagas), y el ΔΦDP sale de medir CADA
celda del perfil de ΦDP con ruido de fase real y desdoblar el perfil
COMPLETO (`np.unwrap`) — medir sólo los dos extremos del tramo no sirve
aquí: el ΔΦDP total de un tramo con varios dB de atenuación supera de sobra
los ±180° que un único par de medidas puede resolver sin ambigüedad.
Promediado sobre `N_REALIZATIONS_PHIDP` desdoblados independientes para leer
el sesgo del método, no el ruido de una sola realización.

In [ ]:
WAVELENGTH_M, PRT_S = 0.10, 1.0e-3
NOISE_FLOOR = 0.05
MEAN_V, SIGMA_V, M = 5.0, 1.5, 128
RADAR_CONSTANT_DB = -20.0
START_RANGE_KM = 5.0
RHO_HV_TRUE = 0.98
N_GATES2 = 60
DR_KM2 = 0.5
R_KM2 = np.arange(N_GATES2) * DR_KM2

z_true_dbz = 25.0 + 25.0 * np.exp(-0.5 * ((R_KM2 - 15.0) / 6.0) ** 2)
alpha_za = 0.0006
a_true = alpha_za * (10.0 ** (z_true_dbz / 10.0)) ** BETA
cum_a_true = np.concatenate([[0.0], np.cumsum(0.5 * (a_true[:-1] + a_true[1:]) * DR_KM2)])
z_attenuated_true_dbz = z_true_dbz - 2.0 * cum_a_true

N_TRIALS_POWER = 300


def measure_dbz(true_dbz, range_km, n_trials, rng):
    power_s = dbz_to_power(true_dbz, range_km, RADAR_CONSTANT_DB)
    powers = []
    for _ in range(n_trials):
        y = generate_cell(power_s, MEAN_V, SIGMA_V, WAVELENGTH_M, PRT_S, M, NOISE_FLOOR, rng)
        r0 = total_power(y)
        n_hat = noise_floor_estimate(y)
        s = subtract_noise(r0, n_hat)
        if s is not None:
            powers.append(s)
    return power_to_dbz(np.mean(powers), range_km, RADAR_CONSTANT_DB)


z_meas_dbz = np.array([
    measure_dbz(z_attenuated_true_dbz[i], START_RANGE_KM + i * DR_KM2, N_TRIALS_POWER, rng)
    for i in range(N_GATES2)
])

kdp_true = a_true / A_COEF_C_BAND
phidp_true = np.concatenate([[0.0], np.cumsum((kdp_true[:-1] + kdp_true[1:]) * DR_KM2)])

N_REALIZATIONS_PHIDP = 20


def simulate_phidp_profile(phidp_true_profile, rng):
    measured = np.array([
        measure_phidp(1.0, MEAN_V, SIGMA_V, WAVELENGTH_M, PRT_S, M, RHO_HV_TRUE, phi, NOISE_FLOOR, rng)
        for phi in phidp_true_profile
    ])
    return np.degrees(np.unwrap(np.radians(measured)))


corrected_sum = np.zeros(N_GATES2)
for _ in range(N_REALIZATIONS_PHIDP):
    phidp_unwrapped = simulate_phidp_profile(phidp_true, rng)
    delta_phidp_measured = phidp_unwrapped[-1] - phidp_unwrapped[0]
    corrected = zphi_correct_dbz(z_meas_dbz, DR_KM2, BETA, A_COEF_C_BAND, delta_phidp_measured)
    corrected_sum += corrected
corrected_mean = corrected_sum / N_REALIZATIONS_PHIDP

interior = slice(5, N_GATES2 - 5)
max_bias_corrected = np.max(np.abs(corrected_mean[interior] - z_true_dbz[interior]))
max_bias_uncorrected = np.max(np.abs(z_meas_dbz[interior] - z_true_dbz[interior]))
BIAS_TOLERANCE_DB = 1.0
print(f"sesgo máximo corregido={max_bias_corrected:.3f} dB, sin corregir={max_bias_uncorrected:.3f} dB")

checks.append((f"caso de modelo acoplado: sesgo corregido < {BIAS_TOLERANCE_DB} dB",
               max_bias_corrected < BIAS_TOLERANCE_DB))
checks.append(("la corrección reduce el sesgo frente a la medida sin corregir",
               max_bias_corrected < max_bias_uncorrected))

## Prueba 3 — robustez numérica ante ΔΦDP degenerado

Con señal detectable en todo el tramo, el denominador de `A(r)` es una
interpolación afín entre dos valores positivos para cualquier `ΔΦDP` real
(ver la página del algoritmo) — no debería haber ningún valor de `ΔΦDP` que
produzca `NaN` o infinito. `ΔΦDP<=0` (ruido de fase sin atenuación real) debe
censurarse a `A=0` en vez de propagar un signo equivocado.

In [ ]:
robust_ok = True
for delta_phidp in [-50, -20, -10, -5, -2, 0.0, 1e-6, 5, 20, 50, 100]:
    a = zphi_specific_attenuation(z_dbz_test, DR_KM, BETA, A_COEF_C_BAND, delta_phidp)
    finite = np.all(np.isfinite(a))
    if delta_phidp <= 0:
        robust_ok = robust_ok and finite and np.allclose(a, 0.0)
    else:
        robust_ok = robust_ok and finite

checks.append(("ninguna combinación produce NaN/infinito; delta_phidp<=0 censura a A=0", robust_ok))
print(f"robusto en toda la malla de delta_phidp probada: {robust_ok}")

## Prueba 4 — sensibilidad a `β` mal asumido (informativa)

**Hallazgo al escribir este oráculo**, no algo anticipado de antemano: con
atenuación total MODERADA, un `β` distinto del real degrada con gracia (sesgo
acotado, siempre mejor que no corregir). Pero con atenuación total ya
severa (decenas de dB — `β_true=0.9` de abajo, un caso extremo pero no
imposible de descartar a priori) el mismo error de `β` puede hacer que la
corrección SOBRECORRIJA y quede peor que publicar la Z sin corregir. No es
pass/fail: es la evidencia que respalda la advertencia de la página del
algoritmo sobre fijar `β`/`a_coef` con cuidado por instalación.

In [ ]:
for beta_true in [0.55, BETA, 0.75, 0.9]:
    a_true_beta = alpha_za * (10.0 ** (z_true_dbz / 10.0)) ** beta_true
    cum_beta = np.concatenate([[0.0], np.cumsum(0.5 * (a_true_beta[:-1] + a_true_beta[1:]) * DR_KM2)])
    z_meas_beta = z_true_dbz - 2.0 * cum_beta
    pia_true = 2.0 * cum_beta[-1]
    dphi = pia_true / A_COEF_C_BAND
    corrected_beta = zphi_correct_dbz(z_meas_beta, DR_KM2, BETA, A_COEF_C_BAND, dphi)
    bias = np.max(np.abs(corrected_beta[interior] - z_true_dbz[interior]))
    bias_uncorrected = np.max(np.abs(z_meas_beta[interior] - z_true_dbz[interior]))
    peor = "PEOR que sin corregir" if bias > bias_uncorrected else "mejor que sin corregir"
    print(f"beta_true={beta_true:.3f}: sesgo corregido={bias:8.3f} dB, sin corregir={bias_uncorrected:8.3f} dB  ({peor})")

checks.append(("informativo: sensibilidad a beta documentada arriba, no es pass/fail", True))

## Prueba 5 — formulación más precisa (banda S, Ecs. 26-27)

**Agregada 2026-09-18**: la instalación de referencia de este repositorio es banda S, y el paper (Tabla 1c,
§c "A more accurate formulation") advierte que la formulación simple de arriba (forzar a 1 el exponente de
la relación KDP-A) no es buena aproximación ahí (`b≈1.18` en banda S, frente a `b≈0.97-0.99` en X/C). Esta
prueba implementa la formulación acorde: resuelve `A(r0)` y el intercepto normalizado de la DSD `N*₀` como
sistema acoplado (Ec. 26 despeja `N*₀` de la relación A-Z en el extremo de referencia; Ec. 27 impone la
restricción de ΔΦDP vía la relación KDP-A, ambas de la Tabla 1 del paper) por bisección — sin forma cerrada,
el paper mismo dice "puede resolverse con una técnica numérica estándar" sin dar una.

**Hallazgo al escribir esta prueba**: `A_true` y `KDP_true` deben derivarse los dos de `Z_true` con el MISMO
`N*₀` (vía las Tablas 1a y 1c) — a diferencia de la formulación simple, que sólo necesita `A_true` y
`ΔΦDP_true` consistentes entre sí vía `a_coef`, sin ninguna relación con `Z_true`. Fijar `A_true` como
constante libre (como hacía el primer intento del test de cableo de `crates/service::ray`, ver
`docs/algorithms/atenuacion-zphi.md`) implica, para la Ec. 26, un `N*₀` absurdo si no coincide con el que la
Ec. 27 asume — el solver converge a una `A(r0)` equivocada.

In [ ]:
# Testud et al. (2000) Tabla 1, canal H, banda S.
BETA_AZ_S = 0.701   # Tabla 1a (relacion A-Z)
A_AZ_S = 9.28e-8    # Tabla 1a
BETA_KDP_S = 1.18   # Tabla 1c (relacion KDP-A)
A_KDP_S = 1.31e3    # Tabla 1c
N0_STAR_MP = 0.8e7  # Marshall-Palmer, mismo valor que usa el paper de referencia


def zphi_solve_accurate(za_dbz, dr_km, beta_az, a_az, beta_kdp, a_kdp, delta_phidp_deg):
    """Resuelve A(r0) y N*_0 (Ecs. 26-27) por biseccion; devuelve el perfil A(r)
    completo via Ec. 19 -- ver docs/algorithms/atenuacion-zphi.md 'Cómo funciona'."""
    za_dbz = np.asarray(za_dbz, dtype=float)
    z_beta = (10.0 ** (za_dbz / 10.0)) ** beta_az
    n = len(z_beta)
    prefix = np.zeros(n)
    for i in range(1, n):
        prefix[i] = prefix[i - 1] + 0.46 * beta_az * 0.5 * (z_beta[i - 1] + z_beta[i]) * dr_km
    i_total = prefix[-1]
    za0_beta = z_beta[-1]
    target_half_dphi = max(delta_phidp_deg, 0.0) / 2.0
    if target_half_dphi == 0.0:
        return np.zeros(n), 0.0, 0.0

    def residual(a0):
        d0 = za0_beta + a0 * i_total
        n0_star = (a0 / (a_az * d0)) ** (1.0 / (1.0 - beta_az))
        tail = i_total - prefix
        denom = za0_beta + a0 * tail
        kernel = (z_beta / denom) ** beta_kdp
        j_integral = np.trapezoid(kernel, dx=dr_km)
        lhs = a_kdp * n0_star ** (1.0 - beta_kdp) * a0 ** beta_kdp * j_integral
        return lhs - target_half_dphi

    lo, hi = 1e-9, 1.0
    f_hi = residual(hi)
    while f_hi < 0.0 and hi < 1e6:
        hi *= 4.0
        f_hi = residual(hi)
    assert f_hi >= 0.0, "no se encontro A(r0) que sature la restriccion de ΔΦDP"
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if residual(mid) > 0.0:
            hi = mid
        else:
            lo = mid
    a0 = 0.5 * (lo + hi)
    d0 = za0_beta + a0 * i_total
    n0_star = (a0 / (a_az * d0)) ** (1.0 / (1.0 - beta_az))
    tail = i_total - prefix
    denom = za0_beta + a0 * tail
    a_profile = a0 * z_beta / denom
    return a_profile, a0, n0_star


def zphi_correct_dbz_accurate(za_dbz, dr_km, beta_az, a_az, beta_kdp, a_kdp, delta_phidp_deg):
    a_profile, _, _ = zphi_solve_accurate(za_dbz, dr_km, beta_az, a_az, beta_kdp, a_kdp, delta_phidp_deg)
    n = len(a_profile)
    corr = np.zeros(n)
    for i in range(1, n):
        corr[i] = corr[i - 1] + (a_profile[i - 1] + a_profile[i]) * dr_km
    return np.asarray(za_dbz, dtype=float) + corr


# Caso de modelo acoplado: A_true y KDP_true se derivan los dos de Z_true con
# el MISMO N*_0 (ver el hallazgo de arriba) -- perfil mas chico que las
# pruebas anteriores porque la atenuacion de banda S para el mismo Z es
# mucho menor que en C/X (ver la tabla de la pagina del algoritmo).
N_S, DR_S = 60, 0.5
R_S = np.arange(N_S) * DR_S
z_true_s_dbz = 20.0 + 25.0 * np.exp(-0.5 * ((R_S - 15.0) / 6.0) ** 2)
a_true_s = A_AZ_S * (N0_STAR_MP ** (1 - BETA_AZ_S)) * (10.0 ** (z_true_s_dbz / 10.0)) ** BETA_AZ_S
cum_a_s = np.concatenate([[0.0], np.cumsum(0.5 * (a_true_s[:-1] + a_true_s[1:]) * DR_S)])
za_meas_s_dbz = z_true_s_dbz - 2.0 * cum_a_s

kdp_true_s = A_KDP_S * (N0_STAR_MP ** (1 - BETA_KDP_S)) * a_true_s ** BETA_KDP_S
cum_kdp_s = np.concatenate([[0.0], np.cumsum(0.5 * (kdp_true_s[:-1] + kdp_true_s[1:]) * DR_S)])
delta_phidp_s_true = 2.0 * cum_kdp_s[-1]

corrected_s = zphi_correct_dbz_accurate(za_meas_s_dbz, DR_S, BETA_AZ_S, A_AZ_S, BETA_KDP_S, A_KDP_S,
                                         delta_phidp_s_true)
interior_s = slice(5, N_S - 5)
bias_accurate_s = np.max(np.abs(corrected_s[interior_s] - z_true_s_dbz[interior_s]))
bias_uncorrected_s = np.max(np.abs(za_meas_s_dbz[interior_s] - z_true_s_dbz[interior_s]))
print(f"banda S, formulacion acorde: sesgo corregido={bias_accurate_s:.4f} dB, "
      f"sin corregir={bias_uncorrected_s:.4f} dB, delta_phidp_true={delta_phidp_s_true:.1f} deg")

# Contraste: la formulacion SIMPLE con los defaults de Gu et al. (2011) para
# banda S sobre la MISMA verdad-terreno -- deberia degradarse claramente,
# tal como advierte el paper (Tabla 1c) para forzar b_KDP=1 en banda S.
A_COEF_S_SIMPLE = 0.02
a_simple_s = zphi_specific_attenuation(za_meas_s_dbz, DR_S, BETA, A_COEF_S_SIMPLE, delta_phidp_s_true)
corr_simple_s = np.concatenate([[0.0], np.cumsum((a_simple_s[:-1] + a_simple_s[1:]) * DR_S)])
ze_simple_s = za_meas_s_dbz + corr_simple_s
bias_simple_s = np.max(np.abs(ze_simple_s[interior_s] - z_true_s_dbz[interior_s]))
print(f"banda S, formulacion SIMPLE (defaults Gu et al. 2011): sesgo corregido={bias_simple_s:.4f} dB")

checks.append(("banda S, formulacion acorde: sesgo corregido < 0.05 dB", bias_accurate_s < 0.05))
checks.append(("banda S, formulacion acorde reduce el sesgo frente a sin corregir",
               bias_accurate_s < bias_uncorrected_s))
checks.append(("banda S: la formulacion simple (Gu et al. 2011) se degrada frente a la acorde",
               bias_simple_s > 5.0 * bias_accurate_s))

## Fuera de alcance, declarado

- **Corrección de atenuación diferencial (ADP) para ZDR**: Py-ART expone
  coeficientes `c`, `d` adicionales para esto (`A_dp = c·A^d`); esta página y
  `crates/attenuation` sólo cubren la atenuación de Z, no la corrección de
  ZDR.
- **Suavizado de Z antes de exponenciar**: Py-ART suaviza la reflectividad
  (`sm_refl`) antes de elevarla a `β` para reducir sensibilidad al ruido de
  medida celda a celda; esta implementación usa la Z ya calibrada/filtrada
  de clutter tal cual, sin paso de suavizado adicional.
- **Estimación adaptativa de `β`/`a_coef`** a partir del propio perfil (en
  vez de una constante por banda fija): variantes publicadas existen, quedan
  fuera de Stage 1 igual que la ventana adaptativa de KDP.
- **Mezcla de hidrometeoros** (granizo con relación Z-atenuación distinta de
  la lluvia) dentro del mismo tramo: el método asume una única relación Z-β
  en todo el tramo contiguo.

In [ ]:
all_ok = True
for name, ok in checks:
    print(f"[{'OK' if ok else 'FALLO'}] {name}")
    all_ok = all_ok and ok

assert all_ok, "el oráculo de atenuación Z-PHI no pasa todas las comprobaciones -- ver tabla arriba"
print("\nTodas las comprobaciones dentro de tolerancia.")